[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/data-engineer-accelerator/blob/main/notebooks/M03_Advanced_SQL_for_DE.ipynb)

# Module 03: Advanced SQL for Data Engineering

**Program:** Quintrix Jr. Data Engineer Training  
**Duration:** 5 hours  
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Database Setup | 10 min |
| 2 | Advanced Aggregations & Pivoting | 40 min |
| 3 | Subqueries — Queries Inside Queries | 20 min |
| 4 | Common Table Expressions (CTEs) | 25 min |
| 5 | Window Functions — The DE Power Tool | 45 min |
| 6 | Date/Time Mastery for Pipelines | 20 min |
| 7 | Data Transformation Patterns | 25 min |
| 8 | Query Optimization & Execution Plans | 40 min |
| 9 | Incremental Loading Strategies | 25 min |
| 10 | SQL Challenge Lab | 30 min |
| 11 | Key Takeaways | 5 min |
| 12 | Homework & Preview | 15 min |

---

## 1. Database Setup

We rebuild the same call center database from M02. Run this cell once — every exercise in this module uses these tables.

**Schema reminder:** `calls` → `orders` → `payments`, with `sources` and `clients` as reference tables. The platform handles both live agent and VA (virtual agent) calls.

| Table | Rows | Key Columns |
|-------|------|-------------|
| `calls` | 40 | call_id, dnis, call_started_at, call_ended_at, disposition, channel |
| `orders` | 15 | order_id, call_id (FK), sku, total |
| `payments` | 15 | transaction_id, order_id (FK), amount, status |
| `sources` | 6 | dnis (PK), campaign_name, client_name, channel |
| `clients` | 3 | client_id, client_name, industry |

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# --- Create tables ---
cursor.execute("""
CREATE TABLE calls (
    call_id TEXT PRIMARY KEY,
    dnis TEXT NOT NULL,
    caller_ani TEXT,
    call_started_at TEXT NOT NULL,
    call_ended_at TEXT,
    disposition TEXT,
    sentiment TEXT,
    channel TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    call_id TEXT,
    sku TEXT NOT NULL,
    subtotal REAL,
    tax REAL,
    shipping REAL,
    total REAL,
    FOREIGN KEY (call_id) REFERENCES calls(call_id)
)
""")

cursor.execute("""
CREATE TABLE payments (
    transaction_id TEXT PRIMARY KEY,
    order_id TEXT,
    auth_code TEXT,
    amount REAL,
    status TEXT,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)
""")

cursor.execute("""
CREATE TABLE sources (
    dnis TEXT PRIMARY KEY,
    campaign_name TEXT NOT NULL,
    client_name TEXT NOT NULL,
    channel TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE clients (
    client_id INTEGER PRIMARY KEY,
    client_name TEXT NOT NULL,
    industry TEXT
)
""")

# --- Insert data ---
clients_data = [
    (1, "Acme Corp", "Retail"),
    (2, "Pinnacle Health", "Healthcare"),
    (3, "Summit Financial", "Financial Services"),
]
cursor.executemany("INSERT INTO clients VALUES (?, ?, ?)", clients_data)

sources_data = [
    ("8005551234", "Acme Spring Sale", "Acme Corp", "live_agent"),
    ("8005551235", "Acme Rewards Program", "Acme Corp", "va"),
    ("8005552345", "Pinnacle Wellness Plan", "Pinnacle Health", "live_agent"),
    ("8005552346", "Pinnacle Rx Refill", "Pinnacle Health", "va"),
    ("8005553456", "Summit Auto Loan", "Summit Financial", "live_agent"),
    ("8005553457", "Summit Credit Card", "Summit Financial", "va"),
]
cursor.executemany("INSERT INTO sources VALUES (?, ?, ?, ?)", sources_data)

calls_data = [
    ("call_001", "8005551234", "3135559876", "2026-03-10T13:00:00Z", "2026-03-10T13:07:45Z", "completed", "positive", "live_agent"),
    ("call_002", "8005551234", "2485555678", "2026-03-10T13:30:00Z", "2026-03-10T13:32:15Z", "dropped", "negative", "live_agent"),
    ("call_003", "8005552345", "7345551234", "2026-03-10T14:00:00Z", "2026-03-10T14:12:30Z", "completed", "neutral", "live_agent"),
    ("call_004", "8005552346", "3135554321", "2026-03-10T14:30:00Z", "2026-03-10T14:35:00Z", "transferred", "neutral", "va"),
    ("call_005", "8005553456", "2485559999", "2026-03-10T15:00:00Z", "2026-03-10T15:08:20Z", "completed", "positive", "live_agent"),
    ("call_006", "8005551235", "5865551111", "2026-03-10T15:30:00Z", "2026-03-10T15:31:10Z", "dropped", "negative", "va"),
    ("call_007", "8005553457", "3135558888", "2026-03-10T16:00:00Z", "2026-03-10T16:06:45Z", "completed", "positive", "va"),
    ("call_008", "8005551234", "7345557777", "2026-03-10T16:30:00Z", "2026-03-10T16:42:00Z", "completed", "neutral", "live_agent"),
    ("call_009", "8005552345", "2485556666", "2026-03-10T17:00:00Z", "2026-03-10T17:01:30Z", "dropped", "negative", "live_agent"),
    ("call_010", "8005553456", "3135553333", "2026-03-10T17:30:00Z", "2026-03-10T17:38:15Z", "completed", "positive", "live_agent"),
    ("call_011", "8005551234", "5865552222", "2026-03-10T18:00:00Z", "2026-03-10T18:09:30Z", "completed", "neutral", "live_agent"),
    ("call_012", "8005552346", "7345554444", "2026-03-10T18:30:00Z", "2026-03-10T18:33:00Z", "transferred", "neutral", "va"),
    ("call_013", "8005551235", "3135551111", "2026-03-10T19:00:00Z", "2026-03-10T19:05:45Z", "completed", "positive", "va"),
    ("call_014", "8005553457", "2485553333", "2026-03-10T19:30:00Z", "2026-03-10T19:31:00Z", "dropped", "negative", "va"),
    ("call_015", "8005551234", "5865554444", "2026-03-10T20:00:00Z", "2026-03-10T20:11:20Z", "completed", "positive", "live_agent"),
    ("call_016", "8005552345", "7345558888", "2026-03-10T20:30:00Z", "2026-03-10T20:36:00Z", "completed", "neutral", "live_agent"),
    ("call_017", "8005553456", "3135559999", "2026-03-10T21:00:00Z", "2026-03-10T21:02:00Z", "dropped", "negative", "live_agent"),
    ("call_018", "8005551235", "2485551111", "2026-03-10T21:30:00Z", "2026-03-10T21:40:15Z", "completed", "neutral", "va"),
    ("call_019", "8005552346", "5865557777", "2026-03-10T22:00:00Z", "2026-03-10T22:04:30Z", "transferred", "neutral", "va"),
    ("call_020", "8005553457", "7345552222", "2026-03-10T22:30:00Z", "2026-03-10T22:37:45Z", "completed", "positive", "va"),
    ("call_021", "8005551234", "3135556666", "2026-03-11T00:00:00Z", "2026-03-11T00:09:30Z", "completed", "neutral", "live_agent"),
    ("call_022", "8005552345", "2485558888", "2026-03-11T00:30:00Z", "2026-03-11T00:31:45Z", "dropped", "negative", "live_agent"),
    ("call_023", "8005553456", "5865553333", "2026-03-11T01:00:00Z", "2026-03-11T01:06:00Z", "completed", "positive", "live_agent"),
    ("call_024", "8005551235", "7345559999", "2026-03-11T01:30:00Z", "2026-03-11T01:38:20Z", "completed", "positive", "va"),
    ("call_025", "8005553457", "3135552222", "2026-03-11T02:00:00Z", "2026-03-11T02:03:15Z", "transferred", "neutral", "va"),
    ("call_026", "8005551234", "2485554444", "2026-03-11T13:00:00Z", "2026-03-11T13:05:30Z", "completed", "positive", "live_agent"),
    ("call_027", "8005552345", "5865556666", "2026-03-11T13:30:00Z", "2026-03-11T13:32:00Z", "dropped", "negative", "live_agent"),
    ("call_028", "8005553456", "7345553333", "2026-03-11T14:00:00Z", "2026-03-11T14:10:45Z", "completed", "neutral", "live_agent"),
    ("call_029", "8005551235", "3135557777", "2026-03-11T14:30:00Z", "2026-03-11T14:34:00Z", "transferred", "neutral", "va"),
    ("call_030", "8005552346", "2485552222", "2026-03-11T15:00:00Z", "2026-03-11T15:08:30Z", "completed", "positive", "va"),
    ("call_031", "8005553457", "5865559999", "2026-03-11T15:30:00Z", "2026-03-11T15:31:20Z", "dropped", "negative", "va"),
    ("call_032", "8005551234", "7345556666", "2026-03-11T16:00:00Z", "2026-03-11T16:07:00Z", "completed", "neutral", "live_agent"),
    ("call_033", "8005552345", "3135554444", "2026-03-11T16:30:00Z", "2026-03-11T16:41:30Z", "completed", "positive", "live_agent"),
    ("call_034", "8005553456", "2485557777", "2026-03-11T17:00:00Z", "2026-03-11T17:01:45Z", "dropped", "negative", "live_agent"),
    ("call_035", "8005551235", "5865558888", "2026-03-11T17:30:00Z", "2026-03-11T17:36:15Z", "completed", "positive", "va"),
    ("call_036", "8005552346", "7345551111", "2026-03-11T18:00:00Z", "2026-03-11T18:03:30Z", "transferred", "neutral", "va"),
    ("call_037", "8005553457", "3135558888", "2026-03-11T18:30:00Z", "2026-03-11T18:39:45Z", "completed", "neutral", "va"),
    ("call_038", "8005551234", "2485559876", "2026-03-11T19:00:00Z", "2026-03-11T19:06:20Z", "completed", "positive", "live_agent"),
    ("call_039", "8005552345", "5865554321", "2026-03-11T19:30:00Z", "2026-03-11T19:32:00Z", "dropped", "negative", "live_agent"),
    ("call_040", "8005553456", "7345559876", "2026-03-11T20:00:00Z", "2026-03-11T20:07:30Z", "completed", "positive", "live_agent"),
]
cursor.executemany("INSERT INTO calls VALUES (?, ?, ?, ?, ?, ?, ?, ?)", calls_data)

orders_data = [
    ("ord_001", "call_001", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_002", "call_003", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_003", "call_005", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_004", "call_008", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_005", "call_010", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_006", "call_011", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_007", "call_015", "SKU-AC-1003", 89.99, 7.20, 5.80, 102.99),
    ("ord_008", "call_016", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_009", "call_020", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
    ("ord_010", "call_021", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_011", "call_023", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_012", "call_026", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_013", "call_030", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_014", "call_033", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_015", "call_040", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
]
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)", orders_data)

payments_data = [
    ("txn_001", "ord_001", "AUTH-7821", 79.99, "approved"),
    ("txn_002", "ord_002", "AUTH-7822", 119.98, "approved"),
    ("txn_003", "ord_003", "AUTH-7823", 49.95, "approved"),
    ("txn_004", "ord_004", "AUTH-7824", 159.97, "approved"),
    ("txn_005", "ord_005", "AUTH-7825", 49.95, "approved"),
    ("txn_006", "ord_006", None, 79.99, "declined"),
    ("txn_007", "ord_007", "AUTH-7827", 102.99, "approved"),
    ("txn_008", "ord_008", "AUTH-7828", 69.95, "approved"),
    ("txn_009", "ord_009", "AUTH-7829", 168.99, "approved"),
    ("txn_010", "ord_010", "AUTH-7830", 79.99, "approved"),
    ("txn_011", "ord_011", "AUTH-7831", 49.95, "approved"),
    ("txn_012", "ord_012", None, 159.97, "declined"),
    ("txn_013", "ord_013", "AUTH-7833", 119.98, "approved"),
    ("txn_014", "ord_014", "AUTH-7834", 69.95, "approved"),
    ("txn_015", "ord_015", "AUTH-7835", 168.99, "approved"),
]
cursor.executemany("INSERT INTO payments VALUES (?, ?, ?, ?, ?)", payments_data)

conn.commit()

# Verify
print("=== Database Ready ===")
for t in ['calls', 'orders', 'payments', 'sources', 'clients']:
    n = cursor.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

---

## 2. Advanced Aggregations & Pivoting

**One-line answer:** GROUP BY gives you totals. HAVING filters those totals. CASE WHEN pivots them into columns.

In M02, you used `GROUP BY`, `COUNT`, and `SUM`. Now we go deeper.

### HAVING vs. WHERE

| Clause | Filters... | When It Runs |
|--------|-----------|-------------|
| `WHERE` | Individual rows BEFORE grouping | Before GROUP BY |
| `HAVING` | Groups AFTER aggregation | After GROUP BY |

```sql
-- WHERE: filter rows, then group
SELECT dnis, COUNT(*) FROM calls WHERE channel = 'va' GROUP BY dnis

-- HAVING: group all rows, then filter groups
SELECT dnis, COUNT(*) FROM calls GROUP BY dnis HAVING COUNT(*) > 5
```

> **Rule:** If you're filtering on a column value, use `WHERE`. If you're filtering on an aggregate (COUNT, SUM, AVG), use `HAVING`.

In [ ]:
# HAVING: Find campaigns with more than 5 calls
print("=== Campaigns with More Than 5 Calls (HAVING) ===")
print(f"{'Campaign':<26} {'Client':<20} {'Calls':>6}")
print("-" * 54)

results = cursor.execute("""
    SELECT s.campaign_name, s.client_name, COUNT(*) as call_count
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    GROUP BY s.campaign_name, s.client_name
    HAVING COUNT(*) > 5
    ORDER BY call_count DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:<20} {row[2]:>6}")

# Compare: WHERE + HAVING together
print("\n=== Completed Calls Per Campaign, Only Campaigns with 3+ Completions ===")
print(f"{'Campaign':<26} {'Completed':>10}")
print("-" * 38)

results = cursor.execute("""
    SELECT s.campaign_name, COUNT(*) as completed
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    WHERE c.disposition = 'completed'          -- WHERE filters rows first
    GROUP BY s.campaign_name
    HAVING COUNT(*) >= 3                        -- HAVING filters groups after
    ORDER BY completed DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:>10}")

print("\nWHERE ran first (kept only completed), then GROUP BY, then HAVING filtered groups.")

### Pivoting with CASE WHEN

SQL doesn't have a built-in PIVOT command in most engines. Instead, you use `CASE WHEN` inside aggregations to turn row values into columns.

```sql
-- Turn disposition values into columns
SELECT campaign_name,
       SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END) as completed,
       SUM(CASE WHEN disposition = 'dropped' THEN 1 ELSE 0 END) as dropped
FROM calls
GROUP BY campaign_name
```

> **Where you'll use this:** BigQuery, PySpark SQL, and reporting queries all use CASE WHEN pivots. dbt models use them constantly.

In [ ]:
# CASE WHEN Pivot: Disposition breakdown by campaign
print("=== Disposition Breakdown by Campaign ===")
print(f"{'Campaign':<26} {'Total':>6} {'Done':>6} {'Drop':>6} {'Xfer':>6} {'Done%':>6}")
print("-" * 58)

results = cursor.execute("""
    SELECT s.campaign_name,
           COUNT(*) as total,
           SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           SUM(CASE WHEN c.disposition = 'dropped' THEN 1 ELSE 0 END) as dropped,
           SUM(CASE WHEN c.disposition = 'transferred' THEN 1 ELSE 0 END) as transferred,
           ROUND(SUM(CASE WHEN c.disposition = 'completed' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) as comp_pct
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    GROUP BY s.campaign_name
    ORDER BY total DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:>6} {row[2]:>6} {row[3]:>6} {row[4]:>6} {row[5]:>5}%")

# Conversion rate by channel
print("\n=== Conversion Rate by Channel ===")
print(f"{'Channel':<12} {'Calls':>6} {'Orders':>7} {'Conv%':>7} {'Revenue':>10}")
print("-" * 44)

results = cursor.execute("""
    SELECT c.channel,
           COUNT(DISTINCT c.call_id) as total_calls,
           COUNT(DISTINCT o.order_id) as orders,
           ROUND(COUNT(DISTINCT o.order_id) * 100.0 / COUNT(DISTINCT c.call_id), 1) as conv_pct,
           ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id
    GROUP BY c.channel
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>7} {row[3]:>6}% ${row[4]:>9.2f}")

print("\nCASE WHEN turns row values into columns — essential for cross-tab reports.")

---

## 3. Subqueries — Queries Inside Queries

**One-line answer:** A subquery is a SELECT inside another SELECT — it lets you use one query's result as input to another.

### Three Types of Subqueries

| Type | Where It Goes | Returns | Example Use |
|------|-------------|---------|-------------|
| **Scalar** | SELECT or WHERE | Single value | Compare to average |
| **Table** | FROM or IN | Set of rows/values | Filter against a list |
| **Correlated** | WHERE | Varies | Row-by-row comparison |

> **Performance note:** Correlated subqueries run once per row in the outer query — they can be slow on large tables. CTEs (Section 4) are often a cleaner alternative.

In [ ]:
# Scalar subquery: Calls longer than average duration
avg_dur = cursor.execute("""
    SELECT ROUND(AVG((julianday(call_ended_at) - julianday(call_started_at)) * 1440), 1)
    FROM calls
""").fetchone()[0]
print(f"Average call duration: {avg_dur} minutes\n")

print("=== Calls Longer Than Average Duration ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14} {'Duration':>8}")
print("-" * 48)

results = cursor.execute("""
    SELECT call_id, channel, disposition,
           ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as dur_min
    FROM calls
    WHERE (julianday(call_ended_at) - julianday(call_started_at)) * 1440 > (
        SELECT AVG((julianday(call_ended_at) - julianday(call_started_at)) * 1440)
        FROM calls
    )
    ORDER BY dur_min DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14} {row[3]:>7.1f}m")
print(f"\n{len(results)} calls above average duration of {avg_dur} min")

# IN subquery: DNIS numbers that had at least one dropped call
print("\n=== Campaigns With Dropped Calls (IN subquery) ===")
results = cursor.execute("""
    SELECT dnis, campaign_name, channel
    FROM sources
    WHERE dnis IN (
        SELECT DISTINCT dnis FROM calls WHERE disposition = 'dropped'
    )
""").fetchall()

for row in results:
    print(f"  {row[1]} ({row[2]}) — DNIS {row[0]}")

In [ ]:
# EXISTS: Completed calls that resulted in approved payments
print("=== Calls With Approved Payments (EXISTS) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14}")
print("-" * 38)

results = cursor.execute("""
    SELECT c.call_id, c.channel, c.disposition
    FROM calls c
    WHERE EXISTS (
        SELECT 1
        FROM orders o
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.call_id = c.call_id
          AND p.status = 'approved'
    )
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14}")
print(f"\n{len(results)} calls with approved payments")

# NOT EXISTS: Completed calls that did NOT result in any order
print("\n=== Completed Calls With NO Order (NOT EXISTS) ===")
results = cursor.execute("""
    SELECT c.call_id, c.channel
    FROM calls c
    WHERE c.disposition = 'completed'
      AND NOT EXISTS (
          SELECT 1 FROM orders o WHERE o.call_id = c.call_id
      )
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"  {row[0]} ({row[1]})")
print(f"\n{len(results)} completed calls had no order — potential missed conversions.")

---

## 4. Common Table Expressions (CTEs)

**One-line answer:** A CTE is a named temporary result set — it makes complex queries readable by breaking them into logical steps.

```sql
WITH step_one AS (
    SELECT ...          -- Calculate something
),
step_two AS (
    SELECT ...          -- Use step_one's result
    FROM step_one
)
SELECT * FROM step_two  -- Final output
```

### CTE vs. Subquery

| Aspect | Subquery | CTE |
|--------|---------|-----|
| Readability | Nested, hard to follow | Named steps, reads top-to-bottom |
| Reuse | Must repeat the subquery | Reference the CTE name multiple times |
| Debugging | Can't run inner query alone easily | Can comment out final SELECT, run CTE alone |
| Performance | Same as CTE in most engines | Same as subquery in most engines |

> **In practice:** Data engineers almost always prefer CTEs over subqueries. dbt models are entirely built on CTEs. BigQuery, Spark SQL, and every modern engine supports them.

In [ ]:
# Basic CTE: Calculate call duration, then filter
print("=== Calls Over 8 Minutes (CTE) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14} {'Duration':>8}")
print("-" * 48)

results = cursor.execute("""
    WITH call_durations AS (
        SELECT call_id, channel, disposition,
               ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as duration_min
        FROM calls
    )
    SELECT call_id, channel, disposition, duration_min
    FROM call_durations
    WHERE duration_min > 8
    ORDER BY duration_min DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14} {row[3]:>7.1f}m")

print(f"\n{len(results)} calls over 8 minutes.")
print("\nThe CTE calculated duration once, then the outer query filtered on it.")
print("Without a CTE, you'd repeat the julianday formula in both SELECT and WHERE.")

In [ ]:
# Multi-CTE: Conversion funnel — Total → Completed → Ordered → Paid
print("=== Call Center Conversion Funnel ===")
print()

result = cursor.execute("""
    WITH total AS (
        SELECT COUNT(*) as n FROM calls
    ),
    completed AS (
        SELECT COUNT(*) as n FROM calls WHERE disposition = 'completed'
    ),
    ordered AS (
        SELECT COUNT(DISTINCT o.call_id) as n
        FROM orders o
    ),
    paid AS (
        SELECT COUNT(DISTINCT o.call_id) as n
        FROM orders o
        JOIN payments p ON o.order_id = p.order_id
        WHERE p.status = 'approved'
    )
    SELECT
        t.n as total_calls,
        c.n as completed,
        o.n as ordered,
        p.n as paid,
        ROUND(c.n * 100.0 / t.n, 1) as comp_pct,
        ROUND(o.n * 100.0 / c.n, 1) as order_pct,
        ROUND(p.n * 100.0 / o.n, 1) as pay_pct
    FROM total t, completed c, ordered o, paid p
""").fetchone()

print(f"  All Calls:       {result[0]:>4}          (100%)")
print(f"  Completed:       {result[1]:>4}  →  {result[4]}% of calls completed")
print(f"  Placed Order:    {result[2]:>4}  →  {result[5]}% of completed placed an order")
print(f"  Payment Approved:{result[3]:>4}  →  {result[6]}% of orders had approved payment")

print("\nEach CTE is one step in the funnel. The final SELECT combines them.")
print("This pattern is how you build analytics funnels in BigQuery, dbt, and dashboards.")

# Funnel by channel
print("\n=== Funnel by Channel ===")
print(f"{'Channel':<12} {'Calls':>6} {'Comp':>6} {'Orders':>7} {'Paid':>6} {'End-to-End':>10}")
print("-" * 49)

results = cursor.execute("""
    WITH funnel AS (
        SELECT c.channel,
               COUNT(DISTINCT c.call_id) as total_calls,
               COUNT(DISTINCT CASE WHEN c.disposition = 'completed' THEN c.call_id END) as completed,
               COUNT(DISTINCT o.order_id) as ordered,
               COUNT(DISTINCT CASE WHEN p.status = 'approved' THEN o.order_id END) as paid
        FROM calls c
        LEFT JOIN orders o ON c.call_id = o.call_id
        LEFT JOIN payments p ON o.order_id = p.order_id
        GROUP BY c.channel
    )
    SELECT channel, total_calls, completed, ordered, paid,
           ROUND(paid * 100.0 / total_calls, 1) as end_to_end_pct
    FROM funnel
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>6} {row[3]:>7} {row[4]:>6} {row[5]:>9}%")

---

## 5. Window Functions — The DE Power Tool

**One-line answer:** Window functions compute values across rows related to the current row — without collapsing the result into groups.

### GROUP BY vs. Window Functions

| Aspect | GROUP BY | Window Function |
|--------|---------|----------------|
| Output rows | One row per group | All original rows preserved |
| Syntax | `GROUP BY column` | `OVER (PARTITION BY ... ORDER BY ...)` |
| Use case | Totals, averages, counts | Rankings, running totals, row comparisons |

### The OVER Clause

```sql
function() OVER (
    PARTITION BY column    -- Optional: split into groups (like GROUP BY, but keeps rows)
    ORDER BY column        -- Optional: define row order within each partition
    ROWS BETWEEN ...       -- Optional: define the window frame
)
```

### Window Functions You'll Use as a DE

| Function | What It Does | DE Use Case |
|----------|-------------|-------------|
| `ROW_NUMBER()` | Unique sequential number per partition | Deduplication — keep only the latest record |
| `RANK()` | Rank with gaps for ties | Leaderboards, top-N analysis |
| `DENSE_RANK()` | Rank without gaps | When you need exactly N distinct ranks |
| `LAG(col, n)` | Value from n rows before | Compare to previous period |
| `LEAD(col, n)` | Value from n rows after | Look ahead (next event) |
| `SUM() OVER` | Running total | Cumulative revenue, progressive counts |
| `AVG() OVER` | Moving average | Smoothed trends, rolling metrics |
| `NTILE(n)` | Split into n equal buckets | Quartile/percentile analysis |

> **Why DEs love window functions:** They solve problems that are painful or impossible with GROUP BY — deduplication, running totals, period-over-period comparisons, and ranking. PySpark has the same window functions with identical syntax.

In [ ]:
# ROW_NUMBER: Number each call within its DNIS
print("=== ROW_NUMBER — Calls Numbered Per DNIS (Acme Spring Sale) ===")
print(f"{'#':>3} {'Call ID':<12} {'Started At':<24} {'Disposition':<14}")
print("-" * 55)

results = cursor.execute("""
    SELECT
        ROW_NUMBER() OVER (PARTITION BY dnis ORDER BY call_started_at) as call_num,
        call_id, call_started_at, disposition
    FROM calls
    WHERE dnis = '8005551234'
    ORDER BY call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:>3} {row[1]:<12} {row[2]:<24} {row[3]:<14}")

print(f"\nROW_NUMBER assigned 1..{len(results)} within this DNIS.")
print("Use case: If call_001 appeared twice, ROW_NUMBER() helps you pick one (deduplication).")

# RANK vs DENSE_RANK: Rank campaigns by revenue
print("\n=== RANK vs DENSE_RANK — Campaigns by Revenue ===")
print(f"{'Campaign':<26} {'Revenue':>10} {'Rank':>5} {'Dense':>6}")
print("-" * 49)

results = cursor.execute("""
    WITH campaign_rev AS (
        SELECT s.campaign_name,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
        FROM sources s
        LEFT JOIN calls c ON s.dnis = c.dnis
        LEFT JOIN orders o ON c.call_id = o.call_id
        GROUP BY s.campaign_name
    )
    SELECT campaign_name, revenue,
           RANK() OVER (ORDER BY revenue DESC) as rnk,
           DENSE_RANK() OVER (ORDER BY revenue DESC) as dense_rnk
    FROM campaign_rev
""").fetchall()

for row in results:
    print(f"{row[0]:<26} ${row[1]:>9.2f} {row[2]:>5} {row[3]:>6}")

### LAG and LEAD — Looking Backward and Forward

`LAG(column, n)` returns the value from `n` rows before the current row.  
`LEAD(column, n)` returns the value from `n` rows after.

```sql
-- Time between consecutive calls on the same DNIS
SELECT call_id,
       call_started_at,
       LAG(call_started_at) OVER (PARTITION BY dnis ORDER BY call_started_at) as prev_call
FROM calls
```

> **DE use case:** LAG is how you calculate period-over-period changes — "How does today's call volume compare to yesterday?" You'll use this pattern in BigQuery marts and PySpark.

In [ ]:
# LAG: Time between consecutive calls on Acme Spring Sale DNIS
print("=== LAG — Time Between Calls (Acme Spring Sale) ===")
print(f"{'Call ID':<12} {'Started At':<24} {'Prev Call':<24} {'Gap':>6}")
print("-" * 68)

results = cursor.execute("""
    SELECT call_id, call_started_at,
           LAG(call_started_at) OVER (ORDER BY call_started_at) as prev_call,
           ROUND(
               (julianday(call_started_at) -
                julianday(LAG(call_started_at) OVER (ORDER BY call_started_at))) * 1440,
               0
           ) as gap_minutes
    FROM calls
    WHERE dnis = '8005551234'
    ORDER BY call_started_at
""").fetchall()

for row in results:
    prev = row[2] if row[2] else '(first call)'
    gap = f"{int(row[3])}m" if row[3] else '-'
    print(f"{row[0]:<12} {row[1]:<24} {prev:<24} {gap:>6}")

# LEAD: What's the NEXT call's disposition?
print("\n=== LEAD — Next Call Disposition ===")
print(f"{'Call ID':<12} {'Disposition':<14} {'Next Disp':<14}")
print("-" * 40)

results = cursor.execute("""
    SELECT call_id, disposition,
           LEAD(disposition) OVER (ORDER BY call_started_at) as next_disposition
    FROM calls
    WHERE dnis = '8005551234'
    ORDER BY call_started_at
""").fetchall()

for row in results:
    next_d = row[2] if row[2] else '(last call)'
    print(f"{row[0]:<12} {row[1]:<14} {next_d:<14}")

### Running Totals and Moving Averages

```sql
-- Running total: sum all rows from the start up to the current row
SUM(amount) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING)

-- Moving average: average of the current row and 2 preceding rows
AVG(amount) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
```

| Frame Clause | Meaning |
|-------------|--------|
| `ROWS UNBOUNDED PRECEDING` | From first row to current row |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | Current row + 2 rows before |
| `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING` | Current row + 1 before + 1 after |

> **DE use case:** Running totals show cumulative revenue. Moving averages smooth out daily spikes in call volume — essential for dashboards and trend analysis.

In [ ]:
# Running total of revenue from orders
print("=== Running Total Revenue ===")
print(f"{'Order':<10} {'Call':<12} {'Amount':>8} {'Running Total':>14} {'Mov Avg (3)':>12}")
print("-" * 58)

results = cursor.execute("""
    SELECT o.order_id, o.call_id, o.total,
           ROUND(SUM(o.total) OVER (ORDER BY c.call_started_at
                ROWS UNBOUNDED PRECEDING), 2) as running_total,
           ROUND(AVG(o.total) OVER (ORDER BY c.call_started_at
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as moving_avg_3
    FROM orders o
    JOIN calls c ON o.call_id = c.call_id
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<10} {row[1]:<12} ${row[2]:>7.2f} ${row[3]:>13.2f} ${row[4]:>11.2f}")

print(f"\nFinal running total: ${results[-1][3]:,.2f}")
print("The moving average smooths out individual order variation.")

In [ ]:
# Campaign performance dashboard using multiple window functions
print("=== Campaign Performance Dashboard (Window Functions) ===")
print(f"{'Campaign':<26} {'Revenue':>9} {'Rev Rank':>9} {'Conv%':>7} {'Conv Rank':>10}")
print("-" * 63)

results = cursor.execute("""
    WITH campaign_stats AS (
        SELECT s.campaign_name, s.channel,
               COUNT(DISTINCT c.call_id) as total_calls,
               COUNT(DISTINCT o.order_id) as orders,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
        FROM calls c
        JOIN sources s ON c.dnis = s.dnis
        LEFT JOIN orders o ON c.call_id = o.call_id
        GROUP BY s.campaign_name, s.channel
    )
    SELECT campaign_name,
           revenue,
           RANK() OVER (ORDER BY revenue DESC) as rev_rank,
           ROUND(orders * 100.0 / total_calls, 1) as conv_rate,
           RANK() OVER (ORDER BY orders * 100.0 / total_calls DESC) as conv_rank
    FROM campaign_stats
    ORDER BY revenue DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} ${row[1]:>8.2f} {row[2]:>9} {row[3]:>6}% {row[4]:>10}")

print("\nNotice: highest revenue doesn't always mean best conversion rate.")
print("Window functions let you rank by multiple criteria in the same query.")

---

## 6. Date/Time Mastery for Pipelines

**One-line answer:** Every pipeline bug you'll encounter involves dates, timezones, or both.

In M02, you saw the UTC bug — evening EST calls appearing as the next day in UTC. Now let's build the reporting queries that handle this correctly.

### SQLite Date/Time Functions

| Function | Example | Result |
|----------|---------|--------|
| `DATE(ts)` | `DATE('2026-03-10T19:00:00Z')` | `2026-03-10` |
| `TIME(ts)` | `TIME('2026-03-10T19:00:00Z')` | `19:00:00` |
| `STRFTIME('%H', ts)` | Extract hour | `19` |
| `STRFTIME('%w', ts)` | Day of week (0=Sun) | `2` |
| `STRFTIME('%Y-%m', ts)` | Year-month | `2026-03` |
| `DATE(ts, '-5 hours')` | UTC to EST | Shifts timestamp |
| `DATE(ts, '+1 day')` | Add 1 day | Next day |
| `julianday(ts)` | Julian day number | For arithmetic |

> **In production:** BigQuery uses `TIMESTAMP_SUB()` and `FORMAT_TIMESTAMP()`. PySpark uses `from_utc_timestamp()`. The concept is the same — always convert to the reporting timezone before aggregating by date.

In [ ]:
# Hourly call volume (EST) — when do calls peak?
print("=== Hourly Call Volume (EST) ===")
print(f"{'Hour (EST)':>10} {'Calls':>6} {'Completed':>10} {'Comp%':>7} {'Bar'}")
print("-" * 50)

results = cursor.execute("""
    SELECT CAST(STRFTIME('%H', call_started_at, '-5 hours') AS INTEGER) as hour_est,
           COUNT(*) as calls,
           SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           ROUND(SUM(CASE WHEN disposition = 'completed' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 0) as comp_pct
    FROM calls
    GROUP BY CAST(STRFTIME('%H', call_started_at, '-5 hours') AS INTEGER)
    ORDER BY hour_est
""").fetchall()

for row in results:
    bar = '█' * row[1]
    ampm = 'AM' if row[0] < 12 else 'PM'
    h = row[0] if row[0] <= 12 else row[0] - 12
    h = 12 if h == 0 else h
    print(f"{h:>6} {ampm}  {row[1]:>6} {row[2]:>10} {row[3]:>6}% {bar}")

print("\nThis is how you build mart_hourly_volume — the gold-layer mart.")
print("Notice: all times converted to EST before grouping.")

In [ ]:
# Daily metrics with proper EST conversion
print("=== Daily Campaign Report (EST) ===")
print(f"{'Date (EST)':<12} {'Calls':>6} {'Comp':>6} {'Orders':>7} {'Revenue':>10} {'Conv%':>7}")
print("-" * 50)

results = cursor.execute("""
    SELECT DATE(c.call_started_at, '-5 hours') as call_date_est,
           COUNT(DISTINCT c.call_id) as calls,
           SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           COUNT(DISTINCT o.order_id) as orders,
           ROUND(COALESCE(SUM(o.total), 0), 2) as revenue,
           ROUND(COUNT(DISTINCT o.order_id) * 100.0 / COUNT(DISTINCT c.call_id), 1) as conv_pct
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id
    GROUP BY DATE(c.call_started_at, '-5 hours')
    ORDER BY call_date_est
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>6} {row[3]:>7} ${row[4]:>9.2f} {row[5]:>6}%")

# Compare UTC vs EST totals to prove the bug
print("\n=== UTC vs EST Daily Totals — Proving the Bug ===")
print(f"{'Date':<12} {'UTC Count':>10} {'EST Count':>10} {'Diff':>6}")
print("-" * 40)

results = cursor.execute("""
    WITH utc_counts AS (
        SELECT DATE(call_started_at) as dt, COUNT(*) as n FROM calls GROUP BY DATE(call_started_at)
    ),
    est_counts AS (
        SELECT DATE(call_started_at, '-5 hours') as dt, COUNT(*) as n FROM calls GROUP BY DATE(call_started_at, '-5 hours')
    )
    SELECT COALESCE(u.dt, e.dt) as dt,
           COALESCE(u.n, 0) as utc_n,
           COALESCE(e.n, 0) as est_n,
           COALESCE(e.n, 0) - COALESCE(u.n, 0) as diff
    FROM utc_counts u
    LEFT JOIN est_counts e ON u.dt = e.dt
    ORDER BY dt
""").fetchall()

for row in results:
    flag = ' ← wrong!' if row[3] != 0 else ''
    print(f"{row[0]:<12} {row[1]:>10} {row[2]:>10} {row[3]:>+6}{flag}")

print("\nAlways convert to reporting timezone BEFORE grouping by date.")

---

## 7. Data Transformation Patterns

**One-line answer:** Transformation is the T in ETL — turning raw data into clean, enriched, analysis-ready data.

### CASE WHEN for Categorization

You already used CASE WHEN for pivoting. It's also how you create derived columns — categorizing raw values into business-meaningful buckets.

```sql
-- Categorize call duration
CASE
    WHEN duration_min < 3 THEN 'Short'
    WHEN duration_min < 8 THEN 'Medium'
    ELSE 'Long'
END as duration_category
```

### NULL Handling

| Function | What It Does | Example |
|----------|-------------|--------|
| `COALESCE(a, b, c)` | First non-NULL value | `COALESCE(auth_code, 'N/A')` |
| `NULLIF(a, b)` | Returns NULL if a = b | `NULLIF(amount, 0)` — prevents divide-by-zero |
| `IFNULL(a, b)` | SQLite-specific COALESCE for 2 args | `IFNULL(sentiment, 'unknown')` |

> **In production:** NULL handling is where most data quality bugs hide. A single NULL in a SUM makes the whole result NULL unless you use COALESCE. Always handle NULLs explicitly.

In [ ]:
# Categorize call duration + revenue tier
print("=== Call Duration Categories ===")
print(f"{'Category':<20} {'Count':>6} {'Avg Duration':>12} {'Completed%':>11}")
print("-" * 51)

results = cursor.execute("""
    WITH call_dur AS (
        SELECT *,
               ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as dur_min
        FROM calls
    )
    SELECT
        CASE
            WHEN dur_min < 3 THEN '1. Short (<3 min)'
            WHEN dur_min < 8 THEN '2. Medium (3-8 min)'
            ELSE '3. Long (>8 min)'
        END as category,
        COUNT(*) as cnt,
        ROUND(AVG(dur_min), 1) as avg_dur,
        ROUND(SUM(CASE WHEN disposition = 'completed' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) as comp_pct
    FROM call_dur
    GROUP BY category
    ORDER BY category
""").fetchall()

for row in results:
    print(f"{row[0]:<20} {row[1]:>6} {row[2]:>10.1f} min {row[3]:>10}%")

print("\nInsight: Longer calls tend to have higher completion rates.")
print("Dropped calls are almost always short — the customer hung up quickly.")

In [ ]:
# NULL handling patterns
print("=== COALESCE: Declined Payments with Default Auth Code ===")
print(f"{'Txn ID':<10} {'Order':<10} {'Auth Code':<12} {'Amount':>8} {'Status':<10}")
print("-" * 52)

results = cursor.execute("""
    SELECT transaction_id, order_id,
           COALESCE(auth_code, '(declined)') as auth_display,
           amount, status
    FROM payments
    WHERE status = 'declined'
""").fetchall()

for row in results:
    print(f"{row[0]:<10} {row[1]:<10} {row[2]:<12} ${row[3]:>7.2f} {row[4]:<10}")

# NULLIF: Safe division
print("\n=== NULLIF: Safe Division (Prevents Divide-by-Zero) ===")
print(f"{'Campaign':<26} {'Calls':>6} {'Orders':>7} {'Conv%':>7}")
print("-" * 48)

results = cursor.execute("""
    SELECT s.campaign_name,
           COUNT(DISTINCT c.call_id) as calls,
           COUNT(DISTINCT o.order_id) as orders,
           ROUND(COUNT(DISTINCT o.order_id) * 100.0 /
                 NULLIF(COUNT(DISTINCT c.call_id), 0), 1) as conv_pct
    FROM sources s
    LEFT JOIN calls c ON s.dnis = c.dnis
    LEFT JOIN orders o ON c.call_id = o.call_id
    GROUP BY s.campaign_name
    ORDER BY conv_pct DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:>6} {row[2]:>7} {row[3]:>6}%")

print("\nNULLIF(x, 0) returns NULL when x is 0, so division yields NULL instead of an error.")
print("COALESCE(result, 0) can then turn that NULL into 0 for display.")

---

## 8. Query Optimization & Execution Plans

**One-line answer:** An execution plan tells you HOW the database will run your query — and where it's wasting time.

### Why DEs Care About Optimization

| Context | Impact of Slow Query |
|---------|--------------------|
| BigQuery | Scans more data = higher cost (pay per byte scanned) |
| PySpark | Unoptimized joins = shuffles = OOM errors |
| Airflow DAG | Slow query = delayed pipeline = stale dashboards |
| Reports | Users waiting 30 seconds vs 2 seconds |

### EXPLAIN QUERY PLAN (SQLite)

Every database has an EXPLAIN command. SQLite's version:

```sql
EXPLAIN QUERY PLAN SELECT * FROM calls WHERE dnis = '8005551234'
```

Key terms in the output:

| Term | Meaning | Good or Bad? |
|------|---------|-------------|
| `SCAN` | Reads every row in the table | Slow on large tables |
| `SEARCH` | Uses an index to find rows | Fast — skips irrelevant rows |
| `USING INDEX` | Which index it's using | Good — means an index exists |
| `USING COVERING INDEX` | Index has all needed columns | Best — doesn't even touch the table |

In [ ]:
# EXPLAIN QUERY PLAN: Before and after indexing
print("=== WITHOUT Index ===")
plan = cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT call_id, disposition FROM calls WHERE dnis = '8005551234'
""").fetchall()
for row in plan:
    print(f"  {row[-1]}")

# Create an index on dnis
cursor.execute("CREATE INDEX idx_calls_dnis ON calls(dnis)")

print("\n=== WITH Index on calls(dnis) ===")
plan = cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT call_id, disposition FROM calls WHERE dnis = '8005551234'
""").fetchall()
for row in plan:
    print(f"  {row[-1]}")

# Index on commonly filtered columns
cursor.execute("CREATE INDEX idx_calls_disposition ON calls(disposition)")
cursor.execute("CREATE INDEX idx_calls_channel ON calls(channel)")
cursor.execute("CREATE INDEX idx_orders_call_id ON orders(call_id)")

print("\n=== JOIN with Index ===")
plan = cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT c.call_id, o.total
    FROM calls c
    JOIN orders o ON c.call_id = o.call_id
    WHERE c.disposition = 'completed'
""").fetchall()
for row in plan:
    print(f"  {row[-1]}")

print("\nSCAN = reads every row (slow on millions of rows)")
print("SEARCH = uses index to jump directly to matching rows (fast)")

### SQL Anti-Patterns That Kill Performance

| Anti-Pattern | Why It's Slow | Better Alternative |
|-------------|--------------|-------------------|
| `SELECT *` | Reads all columns, even unused ones | `SELECT col1, col2` — only what you need |
| `WHERE UPPER(name) = 'ACME'` | Function on column prevents index use | Store normalized, or use case-insensitive collation |
| `WHERE amount != 0` | Negative conditions scan everything | `WHERE amount > 0` — narrower scan |
| `OR` in WHERE | Often prevents index use | `UNION ALL` of two indexed queries |
| `SELECT DISTINCT` on large results | Sorts entire result to deduplicate | Fix the JOIN that's creating duplicates |
| Subquery in SELECT | Runs once per row | JOIN instead |
| `LIKE '%search%'` | Leading wildcard = full scan | Full-text search, or `LIKE 'search%'` |

> **SARGable queries:** A query is "SARGable" (Search ARGument able) when the WHERE clause can use an index. Keep columns bare — don't wrap them in functions.

```sql
-- NOT SARGable (can't use index):
WHERE STRFTIME('%Y', call_started_at) = '2026'

-- SARGable (can use index):
WHERE call_started_at >= '2026-01-01' AND call_started_at < '2027-01-01'
```

In [ ]:
# Performance demonstration with larger dataset
import time
import random

# Generate 10,000 calls for performance testing
cursor.execute("CREATE TABLE calls_large AS SELECT * FROM calls WHERE 1=0")

dnis_list = ['8005551234', '8005551235', '8005552345', '8005552346', '8005553456', '8005553457']
dispositions = ['completed'] * 60 + ['dropped'] * 25 + ['transferred'] * 15
channels = {'8005551234': 'live_agent', '8005551235': 'va', '8005552345': 'live_agent',
            '8005552346': 'va', '8005553456': 'live_agent', '8005553457': 'va'}

large_data = []
for i in range(10000):
    dnis = random.choice(dnis_list)
    hour = random.randint(8, 23)
    minute = random.randint(0, 59)
    day = random.randint(1, 28)
    dur = random.randint(60, 900)  # 1-15 min in seconds
    large_data.append((
        f"lg_{i:05d}", dnis, f"555{random.randint(1000000,9999999)}",
        f"2026-03-{day:02d}T{hour:02d}:{minute:02d}:00Z",
        f"2026-03-{day:02d}T{hour:02d}:{minute + dur//60:02d}:{dur%60:02d}Z",
        random.choice(dispositions),
        random.choice(['positive', 'neutral', 'negative']),
        channels[dnis]
    ))
cursor.executemany("INSERT INTO calls_large VALUES (?, ?, ?, ?, ?, ?, ?, ?)", large_data)
conn.commit()

# Query without index
start = time.time()
for _ in range(100):
    cursor.execute("SELECT COUNT(*) FROM calls_large WHERE dnis = '8005551234'").fetchone()
no_idx = time.time() - start

# Create index
cursor.execute("CREATE INDEX idx_large_dnis ON calls_large(dnis)")

# Query with index
start = time.time()
for _ in range(100):
    cursor.execute("SELECT COUNT(*) FROM calls_large WHERE dnis = '8005551234'").fetchone()
with_idx = time.time() - start

print(f"=== Performance: 10,000 rows, 100 queries ===")
print(f"  Without index: {no_idx:.4f}s")
print(f"  With index:    {with_idx:.4f}s")
print(f"  Speedup:       {no_idx/with_idx:.1f}x faster")
print(f"\nOn 10 million rows, the difference is minutes vs milliseconds.")
print("In BigQuery, it's also dollars — unindexed scans cost more.")

---

## 9. Incremental Loading Strategies

**One-line answer:** Full refresh rebuilds everything every run. Incremental loading only processes what's new.

### Full Refresh vs. Incremental

| Aspect | Full Refresh | Incremental |
|--------|-------------|-------------|
| What it does | DROP + recreate the entire table | INSERT/UPDATE only new or changed rows |
| Speed | Slow on large datasets | Fast — processes only the delta |
| Complexity | Simple — just re-run the query | More complex — need a watermark |
| Data loss risk | Reprocessing errors affect all data | Errors affect only new data |
| When to use | Small tables, early development | Large tables, production pipelines |

### The Watermark Pattern

Track the last successfully processed timestamp. On the next run, only process records newer than that watermark.

```
Run 1: Process all records up to 2026-03-10T23:59:59Z → watermark = 2026-03-10T23:59:59Z
Run 2: Process records WHERE call_started_at > '2026-03-10T23:59:59Z' → watermark = 2026-03-11T20:00:00Z
Run 3: Process records WHERE call_started_at > '2026-03-11T20:00:00Z' → ...
```

> **In production:** The current LT platform uses a full-refresh Windows Service that rebuilds a flat reporting table every 15 minutes. That's the "before" architecture. The "after" (what you'll build) uses incremental loading — only processing new calls since the last run. This is M09 (Delta Lake MERGE) and M11 (Airflow scheduling).

In [ ]:
# Full refresh pattern: Drop and recreate a summary table
print("=== Full Refresh: Daily Summary ===")

cursor.execute("DROP TABLE IF EXISTS daily_summary")
cursor.execute("""
    CREATE TABLE daily_summary AS
    SELECT DATE(c.call_started_at, '-5 hours') as call_date_est,
           COUNT(DISTINCT c.call_id) as total_calls,
           SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           COUNT(DISTINCT o.order_id) as orders,
           ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id
    GROUP BY DATE(c.call_started_at, '-5 hours')
""")

print("daily_summary (full refresh):")
for row in cursor.execute("SELECT * FROM daily_summary ORDER BY call_date_est"):
    print(f"  {row[0]}: {row[1]} calls, {row[2]} completed, {row[3]} orders, ${row[4]:,.2f}")

print("\nFull refresh: simple but rebuilds EVERYTHING every time.")
print("On 10M rows, this takes minutes. Incremental takes seconds.")

In [ ]:
# Incremental loading: Watermark + UPSERT pattern

# Step 1: Simulate a watermark — last processed timestamp
last_watermark = "2026-03-10T22:00:00Z"
print(f"Last watermark: {last_watermark}")
print(f"Only processing calls AFTER this timestamp.\n")

# Step 2: Find new records since watermark
new_calls = cursor.execute("""
    SELECT call_id, dnis, call_started_at, disposition, channel
    FROM calls
    WHERE call_started_at > ?
    ORDER BY call_started_at
""", (last_watermark,)).fetchall()

print(f"=== New Calls Since Watermark: {len(new_calls)} ===")
for row in new_calls[:5]:
    print(f"  {row[0]} | {row[3]:<12} | {row[2]}")
if len(new_calls) > 5:
    print(f"  ... and {len(new_calls) - 5} more")

# Step 3: Update watermark to latest processed record
new_watermark = max(row[2] for row in new_calls)
print(f"\nNew watermark: {new_watermark}")

# Step 4: UPSERT — handle late-arriving data
print("\n=== UPSERT: Handling Late-Arriving Data ===")
print("Scenario: call_002 was marked 'dropped' but the agent actually completed it.")
print("A correction arrives — we need to UPDATE the existing record.\n")

# Show current record
before = cursor.execute("SELECT call_id, disposition FROM calls WHERE call_id = 'call_002'").fetchone()
print(f"Before: {before[0]} → {before[1]}")

# UPSERT using INSERT OR REPLACE
cursor.execute("""
    INSERT OR REPLACE INTO calls
    VALUES ('call_002', '8005551234', '2485555678', '2026-03-10T13:30:00Z',
            '2026-03-10T13:36:45Z', 'completed', 'neutral', 'live_agent')
""")

after = cursor.execute("SELECT call_id, disposition FROM calls WHERE call_id = 'call_002'").fetchone()
print(f"After:  {after[0]} → {after[1]}")

print("\nINSERT OR REPLACE: if the PK exists, replace the row. If not, insert.")
print("In Delta Lake (M09), you'll use MERGE for this — same concept, more control.")
print("In BigQuery, you'll use MERGE ... WHEN MATCHED THEN UPDATE.")

---

## 10. SQL Challenge Lab

**Deliverable:** Complete these 4 challenges. They combine everything from today's module.

Each challenge builds on concepts from multiple sections. Solutions use CTEs, window functions, aggregations, and date handling together.

---

### Challenge 1: Top 3 Campaigns by Conversion Rate

**Task:** Find the top 3 campaigns ranked by conversion rate (orders / total calls). Include total calls, orders, revenue, and the conversion rate. Use the call center's `calls`, `orders`, and `sources` tables.

> This was the homework challenge from M02. Now you have CTEs and window functions to solve it cleanly.

In [ ]:
# Challenge 1: Top 3 campaigns by conversion rate
print("=== Challenge 1: Top 3 Campaigns by Conversion Rate ===")
print(f"{'Rank':>5} {'Campaign':<26} {'Channel':<12} {'Calls':>6} {'Orders':>7} {'Conv%':>7} {'Revenue':>10}")
print("-" * 75)

results = cursor.execute("""
    WITH campaign_metrics AS (
        SELECT s.campaign_name, s.channel,
               COUNT(DISTINCT c.call_id) as total_calls,
               COUNT(DISTINCT o.order_id) as orders,
               ROUND(COUNT(DISTINCT o.order_id) * 100.0 / COUNT(DISTINCT c.call_id), 1) as conv_rate,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
        FROM calls c
        JOIN sources s ON c.dnis = s.dnis
        LEFT JOIN orders o ON c.call_id = o.call_id
        GROUP BY s.campaign_name, s.channel
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY conv_rate DESC) as rnk
        FROM campaign_metrics
    )
    SELECT rnk, campaign_name, channel, total_calls, orders, conv_rate, revenue
    FROM ranked
    WHERE rnk <= 3
""").fetchall()

for row in results:
    print(f"{row[0]:>5} {row[1]:<26} {row[2]:<12} {row[3]:>6} {row[4]:>7} {row[5]:>6}% ${row[6]:>9.2f}")

print("\nCTE → ROW_NUMBER → filter. This is the standard top-N pattern.")

---

### Challenge 2: Daily Revenue with Running Total

**Task:** Show daily revenue (EST) with a running total column. Include call count, order count, and the cumulative revenue across all days.

In [ ]:
# Challenge 2: Daily revenue with running total
print("=== Challenge 2: Daily Revenue with Running Total (EST) ===")
print(f"{'Date (EST)':<12} {'Calls':>6} {'Orders':>7} {'Revenue':>10} {'Running Total':>14}")
print("-" * 51)

results = cursor.execute("""
    WITH daily AS (
        SELECT DATE(c.call_started_at, '-5 hours') as call_date,
               COUNT(DISTINCT c.call_id) as calls,
               COUNT(DISTINCT o.order_id) as orders,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
        FROM calls c
        LEFT JOIN orders o ON c.call_id = o.call_id
        GROUP BY DATE(c.call_started_at, '-5 hours')
    )
    SELECT call_date, calls, orders, revenue,
           ROUND(SUM(revenue) OVER (ORDER BY call_date ROWS UNBOUNDED PRECEDING), 2) as running_total
    FROM daily
    ORDER BY call_date
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>7} ${row[3]:>9.2f} ${row[4]:>13.2f}")

print("\nCTE for daily aggregation → window function for running total.")
print("This is exactly how mart_daily_campaign would be built.")

---

### Challenge 3: Full Campaign Dashboard

**Task:** Build a single query that produces a complete campaign dashboard: calls, completion rate, orders, conversion rate, revenue, average order value, payment approval rate. Rank by revenue.

In [ ]:
# Challenge 3: Full campaign dashboard
print("=== Challenge 3: Campaign Dashboard ===")
print(f"{'Campaign':<24} {'Ch':<4} {'Calls':>5} {'Comp%':>6} {'Ord':>4} {'Conv%':>6} {'Revenue':>9} {'AOV':>7} {'Pay%':>5} {'Rnk':>4}")
print("-" * 80)

results = cursor.execute("""
    WITH dashboard AS (
        SELECT s.campaign_name,
               CASE s.channel WHEN 'live_agent' THEN 'LA' ELSE 'VA' END as ch,
               COUNT(DISTINCT c.call_id) as calls,
               ROUND(SUM(CASE WHEN c.disposition = 'completed' THEN 1.0 ELSE 0 END)
                     / COUNT(DISTINCT c.call_id) * 100, 0) as comp_pct,
               COUNT(DISTINCT o.order_id) as orders,
               ROUND(COUNT(DISTINCT o.order_id) * 100.0
                     / COUNT(DISTINCT c.call_id), 0) as conv_pct,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue,
               ROUND(COALESCE(AVG(o.total), 0), 2) as aov,
               ROUND(COALESCE(
                   SUM(CASE WHEN p.status = 'approved' THEN 1.0 ELSE 0 END)
                   / NULLIF(COUNT(p.transaction_id), 0) * 100, 0), 0) as pay_pct
        FROM calls c
        JOIN sources s ON c.dnis = s.dnis
        LEFT JOIN orders o ON c.call_id = o.call_id
        LEFT JOIN payments p ON o.order_id = p.order_id
        GROUP BY s.campaign_name, s.channel
    )
    SELECT campaign_name, ch, calls, comp_pct, orders, conv_pct,
           revenue, aov, pay_pct,
           RANK() OVER (ORDER BY revenue DESC) as rev_rank
    FROM dashboard
    ORDER BY revenue DESC
""").fetchall()

for row in results:
    pay = f"{int(row[8])}%" if row[8] else 'N/A'
    print(f"{row[0]:<24} {row[1]:<4} {row[2]:>5} {int(row[3]):>5}% {row[4]:>4} {int(row[5]):>5}% ${row[6]:>8.2f} ${row[7]:>6.2f} {pay:>5} {row[9]:>4}")

print("\nThis query joins 4 tables, uses CASE WHEN, COALESCE, NULLIF, and RANK.")
print("In production, this would be a dbt model or BigQuery view.")

---

### Challenge 4: Find Data Anomalies

**Task:** Write queries to find data quality issues:
1. Calls where channel doesn't match their DNIS source mapping
2. Orders without matching calls (orphaned records)
3. Calls with zero or negative duration
4. Duplicate caller_ani values (same person calling multiple times)

In [ ]:
# Challenge 4: Data anomaly detection
print("=== Challenge 4: Data Anomaly Detection ===\n")

# Check 1: Channel mismatches
mismatches = cursor.execute("""
    SELECT c.call_id, c.channel as call_channel, s.channel as source_channel
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    WHERE c.channel != s.channel
""").fetchall()
print(f"1. Channel mismatches (call vs source): {len(mismatches)}")
if mismatches:
    for row in mismatches:
        print(f"   {row[0]}: call={row[1]}, source={row[2]}")
else:
    print("   None found — data is consistent.")

# Check 2: Orphaned orders (no matching call)
orphans = cursor.execute("""
    SELECT o.order_id, o.call_id
    FROM orders o
    LEFT JOIN calls c ON o.call_id = c.call_id
    WHERE c.call_id IS NULL
""").fetchall()
print(f"\n2. Orphaned orders (no matching call): {len(orphans)}")
if orphans:
    for row in orphans:
        print(f"   {row[0]} → references missing {row[1]}")
else:
    print("   None found — all orders link to valid calls.")

# Check 3: Zero or negative duration
bad_duration = cursor.execute("""
    SELECT call_id,
           ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as dur_min
    FROM calls
    WHERE julianday(call_ended_at) <= julianday(call_started_at)
""").fetchall()
print(f"\n3. Zero or negative duration calls: {len(bad_duration)}")
if bad_duration:
    for row in bad_duration:
        print(f"   {row[0]}: {row[1]} min")
else:
    print("   None found — all calls have positive duration.")

# Check 4: Repeat callers (same phone number, multiple calls)
print(f"\n4. Repeat callers (same phone, multiple calls):")
repeats = cursor.execute("""
    SELECT caller_ani, COUNT(*) as call_count,
           GROUP_CONCAT(DISTINCT disposition) as dispositions
    FROM calls
    GROUP BY caller_ani
    HAVING COUNT(*) > 1
    ORDER BY call_count DESC
    LIMIT 5
""").fetchall()

for row in repeats:
    print(f"   {row[0]}: {row[1]} calls ({row[2]})")

print(f"\n   {len(repeats)} repeat callers found.")
print("   In production, repeat callers could be: callbacks, complaints, or duplicates.")
print("\nThese 4 checks are the foundation of data quality testing (M14).")

---

## 11. Key Takeaways

1. **HAVING filters groups; WHERE filters rows.** Use HAVING for conditions on aggregates (COUNT, SUM, AVG).
2. **CASE WHEN pivots rows into columns.** Essential for cross-tab reports — no PIVOT command needed.
3. **Subqueries nest queries inside queries.** Useful but can be hard to read — CTEs are usually better.
4. **CTEs break complex queries into named steps.** Read top-to-bottom, reusable, debuggable. dbt is built entirely on CTEs.
5. **Window functions are the DE power tool.** ROW_NUMBER for deduplication, RANK for top-N, LAG/LEAD for period comparison, SUM OVER for running totals.
6. **Always convert to reporting timezone before grouping by date.** The UTC bug is real — evening EST calls appear as the next day in UTC.
7. **COALESCE and NULLIF prevent silent failures.** NULL in a SUM = NULL result. Always handle NULLs explicitly.
8. **EXPLAIN shows how the database executes your query.** SCAN = slow, SEARCH = fast. Indexes turn scans into searches.
9. **Incremental loading processes only what's new.** Watermark pattern tracks the last processed timestamp. Full refresh is simple but doesn't scale.
10. **Data quality checks are SQL queries.** Orphaned records, mismatches, duplicates, impossible values — all found with LEFT JOIN + IS NULL, GROUP BY + HAVING, and CASE WHEN.
11. **These SQL patterns transfer directly to PySpark (M06), BigQuery (M08), and dbt.** The syntax varies slightly but the concepts are identical.

---

## 12. Homework & Preview

### 1. SQL Practice (30 min)

Write these queries using the call center database:

a. **Top 3 hours of the day (EST) by call volume.** Include completion rate per hour.  
b. **Revenue by client with running total.** Use a CTE + window function.  
c. **Find all calls that were completed but had a declined payment.** What's the business impact?

### 2. SQLBolt Lessons 13-18 (30 min)

Complete [SQLBolt](https://sqlbolt.com/) Lessons 13-18 — INSERT, UPDATE, DELETE, CREATE TABLE, ALTER TABLE, DROP TABLE.

### 3. Challenge: The Conversion Funnel by Channel

Build a single CTE query that shows the full conversion funnel (calls → completed → ordered → paid) broken down by channel (live_agent vs va). Which channel has better end-to-end conversion?

### 4. Preview M04: Data Modeling

Skim these concepts — we'll cover them in the next session:
- **Star schema** — fact tables vs dimension tables
- **Slowly Changing Dimensions (SCD Type 2)** — how to track history
- **Partitioning** — splitting large tables for performance

Think about: In our call center data, which table is the fact table? Which are dimensions?

---

**Next session:** M04 — Data Modeling (star schema, SCD Type 2, partitioning)